# Acquire a source file (F04.2 + F04.3)

Validate a staged source file and land it immutably under the managed
source Volume. This notebook is acquisition-only: it never parses
Airbnb records and never writes Bronze or Silver tables.

## Run modes (F04.3)

* `acquire` - validate the staged file, land it immutably and register
  the attempt in the manifest table (`registered` -> `landed`, or
  `failed` with the sanitized reason).
* `verify_existing` - hash-verify the landed object for a known
  `source_file_id` (or identity + hash) without touching `_incoming`.
* `reprocess` - resolve the immutable landing path for an identity from
  the manifest table; no download, no copy. Later Bronze notebooks
  consume this `landing_path` as their input.
* `inspect` - show the stored manifest record for an ID.

The manifest table (default `trippilot_dev.ops.source_file_manifest`)
stores one row per deterministic `source_file_id`; writes are
idempotent Delta MERGEs that update only operational fields. Isolated
experiments and runtime tests use
`trippilot_dev.ops.source_file_manifest_test`.

## Manual upload route (no outbound network needed)

1. Download the source file on any machine with internet access.
2. In Databricks, open Catalog Explorer -> `trippilot_dev` -> `bronze` ->
   `Volumes` -> `source_files`.
3. Open `_incoming` (create `_incoming/<source_family>/<entity>`
   subfolders as needed) and use **Upload to this volume**.
4. Run this notebook with the staged path in the widget below.

Landing is idempotent: re-running with the same bytes verifies the
landed object and returns the existing record. A corrected file
(different SHA-256) lands as a separate hash-versioned object; nothing
is ever overwritten.

In [0]:
# Source identity and provenance inputs (F04.2-T01).
dbutils.widgets.text(
    "source_path",
    "/Volumes/trippilot_dev/bronze/source_files/_incoming/insideairbnb/toronto/listings.csv.gz",
    "Staged source path (under _incoming)",
)
dbutils.widgets.text("source_family", "insideairbnb", "Source family")
dbutils.widgets.text("source_entity_id", "toronto", "Source entity ID (city, agency, document)")
dbutils.widgets.text("source_version", "2026-09-16", "Source version or snapshot")
dbutils.widgets.text("file_type", "csv.gz", "File type")
dbutils.widgets.text("source_url", "https://insideairbnb.com/get-the-data/", "Official source URL")
dbutils.widgets.text("schema_version", "pending", "Schema version")
dbutils.widgets.text("license", "CC0-1.0", "Licence")
dbutils.widgets.text(
    "attribution", "Inside Airbnb (insideairbnb.com), CC0 1.0", "Attribution"
)
dbutils.widgets.text(
    "retrieved_at_utc", "", "Retrieved at (UTC, e.g. 2026-11-13T18:00:00Z; blank = now)"
)

# Run-mode and manifest-store controls (F04.3-T01).
dbutils.widgets.dropdown(
    "run_mode",
    "acquire",
    ["acquire", "verify_existing", "reprocess", "inspect"],
    "Run mode",
)
dbutils.widgets.text(
    "manifest_table",
    "trippilot_dev.ops.source_file_manifest",
    "Manifest table",
)
dbutils.widgets.text(
    "source_file_id",
    "",
    "Known source_file_id (verify/inspect; blank = derive from identity)",
)
dbutils.widgets.text(
    "file_hash_sha256",
    "",
    "SHA-256 for verify/reprocess by identity (blank = use source_file_id)",
)

In [0]:
# Locate the repository root and import the shared acquisition module.
from datetime import UTC, datetime
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config/project.yml").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate repository root. Open this notebook inside the Databricks Git folder."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

from src.common.source_acquisition import (
    acquire_source_file,
    reprocess_source_file,
    verify_existing_source_file,
)
from src.common.source_manifest import (
    build_source_file_id,
    compute_file_hash_sha256,
)
from src.common.source_manifest_store import ManifestStore

SOURCE_ROOT = "/Volumes/trippilot_dev/bronze/source_files"

In [0]:
# Build the manifest metadata from the widgets and land the file.
source_path = dbutils.widgets.get("source_path")

retrieved_at = dbutils.widgets.get("retrieved_at_utc")
if not retrieved_at:
    retrieved_at = datetime.now(UTC).isoformat().replace("+00:00", "Z")

source_metadata = {
    "source_family": dbutils.widgets.get("source_family"),
    "source_entity_id": dbutils.widgets.get("source_entity_id"),
    "source_version": dbutils.widgets.get("source_version"),
    "file_type": dbutils.widgets.get("file_type"),
    "source_url": dbutils.widgets.get("source_url"),
    "retrieved_at_utc": retrieved_at,
    "schema_version": dbutils.widgets.get("schema_version"),
    "license": dbutils.widgets.get("license"),
    "attribution": dbutils.widgets.get("attribution"),
    "source_path": source_path,
}

# For run_mode=acquire these inputs feed acquire_source_file; the
# dispatch cell below performs the selected run-mode action.

In [0]:
# Run-mode dispatch (F04.3-T01): explicit action controls.
#
# acquire        - validate + land the staged file and register the
#                  attempt in the manifest table.
# verify_existing - hash-verify the landed object for a known
#                  source_file_id (or identity + hash) without
#                  touching the incoming staging area.
# reprocess      - resolve the immutable landing path for an identity
#                  from the manifest table; no download, no copy.
# inspect        - show the stored manifest record for an ID.
run_mode = dbutils.widgets.get("run_mode")

manifest_store = ManifestStore(
    table_name=dbutils.widgets.get("manifest_table"), spark=spark
)
manifest_store.ensure_table()


def _resolve_source_file_id():
    """Prefer an explicit source_file_id; else derive it from identity."""
    known_id = dbutils.widgets.get("source_file_id").strip()
    if known_id:
        return known_id
    file_hash = dbutils.widgets.get("file_hash_sha256").strip()
    if not file_hash:
        raise ValueError(
            "verify_existing needs the source_file_id widget, or the "
            "identity widgets plus file_hash_sha256"
        )
    return build_source_file_id(
        dbutils.widgets.get("source_family"),
        dbutils.widgets.get("source_entity_id"),
        dbutils.widgets.get("source_version"),
        dbutils.widgets.get("file_type"),
        file_hash,
    )


def _identity_with_hash():
    """Build the reprocess identity from the widgets."""
    file_hash = dbutils.widgets.get("file_hash_sha256").strip()
    if not file_hash:
        raise ValueError("reprocess requires the file_hash_sha256 widget")
    return {
        "source_family": dbutils.widgets.get("source_family"),
        "source_entity_id": dbutils.widgets.get("source_entity_id"),
        "source_version": dbutils.widgets.get("source_version"),
        "file_type": dbutils.widgets.get("file_type"),
        "file_hash_sha256": file_hash,
    }


if run_mode == "acquire":
    record = acquire_source_file(
        source_metadata, source_path, SOURCE_ROOT, manifest_store=manifest_store
    )
elif run_mode == "verify_existing":
    record = verify_existing_source_file(_resolve_source_file_id(), manifest_store)
elif run_mode == "reprocess":
    record = reprocess_source_file(_identity_with_hash(), manifest_store)
elif run_mode == "inspect":
    known_id = dbutils.widgets.get("source_file_id").strip()
    if not known_id:
        raise ValueError("inspect requires the source_file_id widget")
    record = manifest_store.read(known_id)
    if record is None:
        raise ValueError(f"{known_id}: no manifest record found")
else:
    raise ValueError(f"Unknown run_mode {run_mode!r}")

print(f"run_mode selected: {run_mode}")

In [0]:
# Result: the manifest record for the selected run mode (F04.1 contract).
print(f"run_mode         : {run_mode}")
print(f"source_file_id   : {record['source_file_id']}")
print(f"processing_state : {record['processing_state']}")
print(f"run_id           : {record['run_id']}")
print(f"byte_size        : {record['byte_size']}")
print(f"file_hash_sha256 : {record['file_hash_sha256']}")

if record.get("landing_path"):
    landed_hash = compute_file_hash_sha256(record["landing_path"])
    assert landed_hash == record["file_hash_sha256"], "Landed file hash mismatch"
    print(f"landing_path     : {record['landing_path']} (hash verified)")
    if run_mode == "reprocess":
        print(
            "Use landing_path as the Bronze input: reprocessing never "
            "re-downloads or re-copies the source file."
        )

if record.get("error_message"):
    print(f"error_message    : {record['error_message']}")